In [1]:
import subprocess
import glob
import os
import re
import struct
import pandas as pd

KTEST_TOOL = "/mnt/data2/EMULATOR/misc/klee/build/bin/ktest-tool"
OUT_DIR = "out_spike_satp"

def parse_ktest(path):
    out = subprocess.check_output([KTEST_TOOL, path], text=True, errors="ignore")
    data = {}
    current_name = None

    for line in out.splitlines():
        line = line.strip()

        m = re.match(r"object \d+: name: '(.+)'", line)
        if m:
            current_name = m.group(1)
            continue

        m = re.match(r"object \d+: hex\s*:\s*0x([0-9a-fA-F]+)", line)
        if m and current_name:
            data[current_name] = bytes.fromhex(m.group(1))
            continue

    return data

def read_i32(raw):
    if raw is None:
        return None
    return struct.unpack("<i", raw[:4])[0]

def mode_name(x):
    return {
        0: "OFF",
        8: "SV39",
        9: "SV48",
        10: "SV57",
        11: "SV64",
        15: "RESERVED",
    }.get(x, f"UNKNOWN_{x}")

def prv_name(x):
    return {
        1: "S",
        3: "M",
    }.get(x, f"UNKNOWN_{x}")

def exc_name(x):
    if x == 0:
        return "none"
    if x == 2:
        return "illegal instruction"
    if x == 22:
        return "virtual instruction"
    return f"unknown {x}"

def satp_mode_valid(mode, max_vaddr_bits):
    if mode == 0:
        return True
    if mode == 8 and max_vaddr_bits >= 39:
        return True
    if mode == 9 and max_vaddr_bits >= 48:
        return True
    if mode == 10 and max_vaddr_bits >= 57:
        return True
    return False

def explain(row):
    target = "VSATP" if row["obs_target_is_vsatp"] == 1 else "SATP"

    if row["obs_exc"] == 22:
        return "Virtualized SATP access with hstatus.VTVM set raises a virtual instruction exception; no SATP or VSATP write occurs and no TLB flush occurs."

    if row["obs_exc"] == 2:
        return "Non virtual SATP access with mstatus.TVM set below M privilege raises an illegal instruction exception; no SATP write occurs and no TLB flush occurs."

    if row["obs_has_mmu"] == 0:
        if row["obs_tlb_flushed"] == 1:
            return f"{target} write succeeds when MMU is unavailable; the written ATP value is forced to zero, and because the resulting value differs from the old value, the TLB is flushed."
        return f"{target} write succeeds when MMU is unavailable; the written ATP value is forced to zero, and because the resulting value equals the old value, the TLB is not flushed."

    write_mode = row["obs_write_mode"]
    max_bits = row["obs_max_vaddr_bits"]

    if satp_mode_valid(write_mode, max_bits):
        mode_part = f"the written mode {mode_name(write_mode)} is valid for max_vaddr_bits={max_bits} and is accepted"
    else:
        mode_part = f"the written mode {mode_name(write_mode)} is invalid or unsupported for max_vaddr_bits={max_bits}, so the previous mode is preserved"

    if row["obs_impl_mmu_asid"] == 0:
        asid_part = "ASID support is disabled, so the written ASID is masked to zero"
    else:
        asid_part = "ASID support is enabled, so the written ASID is preserved within the implemented ASID mask"

    if row["obs_paddr_bits"] == 12:
        ppn_part = "paddr_bits=12 permits no modeled PPN bits, so the written PPN is masked to zero"
    elif row["obs_paddr_bits"] == 13:
        ppn_part = "paddr_bits=13 permits one modeled PPN bit, so the written PPN is masked to one bit"
    else:
        ppn_part = "paddr_bits=14 permits two modeled PPN bits, so the written PPN is masked to two bits"

    if row["obs_tlb_flushed"] == 1:
        flush_part = "the resulting ATP value differs from the old value, so the TLB is flushed"
    else:
        flush_part = "the resulting ATP value equals the old value, so the TLB is not flushed"

    return f"{target} write succeeds; {mode_part}; {asid_part}; {ppn_part}; {flush_part}."

names = [
    "has_mmu_idx",
    "v_idx",
    "prv_idx",
    "hstatus_vtvm_idx",
    "mstatus_tvm_idx",
    "impl_mmu_asid_idx",
    "max_vaddr_bits_idx",
    "paddr_bits_idx",

    "satp_old_mode_idx",
    "satp_old_asid",
    "satp_old_ppn",

    "vsatp_old_mode_idx",
    "vsatp_old_asid",
    "vsatp_old_ppn",

    "write_mode_idx",
    "write_asid",
    "write_ppn",

    "obs_has_mmu",
    "obs_v",
    "obs_prv",
    "obs_hstatus_vtvm",
    "obs_mstatus_tvm",
    "obs_impl_mmu_asid",
    "obs_max_vaddr_bits",
    "obs_paddr_bits",

    "obs_satp_old_mode",
    "obs_vsatp_old_mode",
    "obs_write_mode",

    "obs_exc",
    "obs_tlb_flushed",
    "obs_target_is_vsatp",

    "obs_satp_new_mode",
    "obs_satp_new_asid",
    "obs_satp_new_ppn",

    "obs_vsatp_new_mode",
    "obs_vsatp_new_asid",
    "obs_vsatp_new_ppn",
]

rows = []

for path in sorted(glob.glob(f"{OUT_DIR}/test*.ktest")):
    d = parse_ktest(path)
    row = {"ktest": os.path.basename(path)}

    for name in names:
        row[name] = read_i32(d.get(name))

    rows.append(row)

df = pd.DataFrame(rows)

df["exc_name"] = df["obs_exc"].apply(exc_name)
df["prv_name"] = df["obs_prv"].apply(prv_name)
df["satp_old_mode_name"] = df["obs_satp_old_mode"].apply(mode_name)
df["vsatp_old_mode_name"] = df["obs_vsatp_old_mode"].apply(mode_name)
df["write_mode_name"] = df["obs_write_mode"].apply(mode_name)
df["satp_new_mode_name"] = df["obs_satp_new_mode"].apply(mode_name)
df["vsatp_new_mode_name"] = df["obs_vsatp_new_mode"].apply(mode_name)

df["target"] = df["obs_target_is_vsatp"].apply(lambda x: "VSATP" if x == 1 else "SATP")
df["path_summary"] = df.apply(explain, axis=1)

df = df.sort_values([
    "obs_exc",
    "obs_v",
    "obs_hstatus_vtvm",
    "obs_mstatus_tvm",
    "obs_prv",
    "obs_has_mmu",
    "obs_write_mode",
    "obs_impl_mmu_asid",
    "obs_paddr_bits",
    "obs_tlb_flushed",
    "ktest",
])

df.to_csv("spike_satp_klee_paths_raw.csv", index=False)

group_cols = [
    "target",
    "exc_name",
    "obs_tlb_flushed",
    "obs_has_mmu",
    "obs_v",
    "prv_name",
    "obs_hstatus_vtvm",
    "obs_mstatus_tvm",
    "obs_impl_mmu_asid",
    "obs_max_vaddr_bits",
    "obs_paddr_bits",
    "satp_old_mode_name",
    "vsatp_old_mode_name",
    "write_mode_name",
    "satp_new_mode_name",
    "vsatp_new_mode_name",
    "path_summary",
]

g = (
    df.groupby(group_cols, dropna=False)
      .size()
      .reset_index(name="path_count")
      .sort_values(["exc_name", "target", "path_count"], ascending=[True, True, False])
)

g.to_csv("spike_satp_klee_path_summary_groups.csv", index=False)

print("raw ktests:", len(df))
print("unique grouped summaries:", len(g))
print()
print("Exception counts:")
print(df["exc_name"].value_counts().to_string())
print()
print("Target counts:")
print(df["target"].value_counts().to_string())
print()
print("TLB flush counts:")
print(df["obs_tlb_flushed"].value_counts().to_string())
print()
print("First 30 grouped summaries:")
print(g[["path_count", "target", "exc_name", "obs_tlb_flushed", "path_summary"]].head(30).to_string(index=False))

raw ktests: 29232
unique grouped summaries: 13326

Exception counts:
exc_name
none                   26640
virtual instruction     1728
illegal instruction      864

Target counts:
target
SATP     16848
VSATP    12384

TLB flush counts:
obs_tlb_flushed
1    19800
0     9432

First 30 grouped summaries:
 path_count target            exc_name  obs_tlb_flushed                                                                                                                                          path_summary
          9   SATP illegal instruction                0 Non virtual SATP access with mstatus.TVM set below M privilege raises an illegal instruction exception; no SATP write occurs and no TLB flush occurs.
          9   SATP illegal instruction                0 Non virtual SATP access with mstatus.TVM set below M privilege raises an illegal instruction exception; no SATP write occurs and no TLB flush occurs.
          9   SATP illegal instruction                0 Non virtual SATP acces

In [2]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("spike_satp_klee_paths_raw.csv")

print(len(df))
print(df["ktest"].nunique())
print(df.isna().sum().sort_values(ascending=False).head(20))

29232
29232
has_mmu_idx           29232
v_idx                 29232
prv_idx               29232
mstatus_tvm_idx       29232
hstatus_vtvm_idx      29232
impl_mmu_asid_idx     29232
max_vaddr_bits_idx    29232
satp_old_mode_idx     29232
paddr_bits_idx        29232
satp_old_asid         29232
ktest                     0
satp_old_ppn              0
vsatp_old_mode_idx        0
vsatp_old_asid            0
vsatp_old_ppn             0
write_mode_idx            0
write_asid                0
write_ppn                 0
obs_has_mmu               0
obs_v                     0
dtype: int64


In [3]:
import subprocess
import re

KTEST_TOOL = "/mnt/data2/EMULATOR/misc/klee/build/bin/ktest-tool"
out = subprocess.check_output(
    [KTEST_TOOL, "out_spike_satp/test000001.ktest"],
    text=True,
    errors="ignore"
)

names = re.findall(r"object \d+: name: '(.+)'", out)
print("\n".join(names))

satp_old_ppn
vsatp_old_mode_idx
vsatp_old_asid
vsatp_old_ppn
write_mode_idx
write_asid
write_ppn
obs_has_mmu
obs_v
obs_prv
obs_hstatus_vtvm
obs_mstatus_tvm
obs_impl_mmu_asid
obs_max_vaddr_bits
obs_paddr_bits
obs_satp_old_mode
obs_vsatp_old_mode
obs_write_mode
obs_exc
obs_tlb_flushed
obs_target_is_vsatp
obs_satp_new_mode
obs_satp_new_asid
obs_satp_new_ppn
obs_vsatp_new_mode
obs_vsatp_new_asid
obs_vsatp_new_ppn


In [4]:
import pandas as pd

df = pd.read_csv("spike_satp_klee_paths_raw.csv")

drop_cols = [
    "has_mmu_idx",
    "v_idx",
    "prv_idx",
    "hstatus_vtvm_idx",
    "mstatus_tvm_idx",
    "impl_mmu_asid_idx",
    "max_vaddr_bits_idx",
    "paddr_bits_idx",
    "satp_old_mode_idx",
]

df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

df_clean.to_csv("spike_satp_klee_paths_raw_clean.csv", index=False)

print(len(df_clean))
print(df_clean["ktest"].nunique())
print(df_clean.isna().sum().sort_values(ascending=False).head(30))

29232
29232
satp_old_asid          29232
ktest                      0
satp_old_ppn               0
vsatp_old_mode_idx         0
vsatp_old_asid             0
vsatp_old_ppn              0
write_mode_idx             0
write_asid                 0
write_ppn                  0
obs_has_mmu                0
obs_v                      0
obs_prv                    0
obs_hstatus_vtvm           0
obs_mstatus_tvm            0
obs_impl_mmu_asid          0
obs_max_vaddr_bits         0
obs_paddr_bits             0
obs_satp_old_mode          0
obs_vsatp_old_mode         0
obs_write_mode             0
obs_exc                    0
obs_tlb_flushed            0
obs_target_is_vsatp        0
obs_satp_new_mode          0
obs_satp_new_asid          0
obs_satp_new_ppn           0
obs_vsatp_new_mode         0
obs_vsatp_new_asid         0
obs_vsatp_new_ppn          0
exc_name                   0
dtype: int64


In [5]:
df_clean = pd.read_csv("spike_satp_klee_paths_raw_clean.csv")

drop_cols = [c for c in ["satp_old_asid"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=drop_cols)

df_clean.to_csv("spike_satp_klee_paths_raw_clean.csv", index=False)

print(len(df_clean))
print(df_clean["ktest"].nunique())
print(df_clean.isna().sum().sort_values(ascending=False).head(30))

29232
29232
ktest                  0
satp_old_ppn           0
vsatp_old_mode_idx     0
vsatp_old_asid         0
vsatp_old_ppn          0
write_mode_idx         0
write_asid             0
write_ppn              0
obs_has_mmu            0
obs_v                  0
obs_prv                0
obs_hstatus_vtvm       0
obs_mstatus_tvm        0
obs_impl_mmu_asid      0
obs_max_vaddr_bits     0
obs_paddr_bits         0
obs_satp_old_mode      0
obs_vsatp_old_mode     0
obs_write_mode         0
obs_exc                0
obs_tlb_flushed        0
obs_target_is_vsatp    0
obs_satp_new_mode      0
obs_satp_new_asid      0
obs_satp_new_ppn       0
obs_vsatp_new_mode     0
obs_vsatp_new_asid     0
obs_vsatp_new_ppn      0
exc_name               0
prv_name               0
dtype: int64


In [6]:
from pathlib import Path
import re

INPUT_FILE = Path("summary.txt")
OUT1 = Path("summary_1.txt")
OUT2 = Path("summary_2.txt")

text = INPUT_FILE.read_text(errors="ignore")

blocks = re.split(r"(?=^Path\s+\d+\s*:)", text, flags=re.MULTILINE)
prefix = ""

if blocks and not re.match(r"^Path\s+\d+\s*:", blocks[0].lstrip()):
    prefix = blocks[0]
    blocks = blocks[1:]

blocks = [b for b in blocks if b.strip()]

n = len(blocks)

if n == 0:
    lines = text.splitlines(keepends=True)
    mid = (len(lines) + 1) // 2
    OUT1.write_text("".join(lines[:mid]))
    OUT2.write_text("".join(lines[mid:]))
    print("No Path blocks found. Split by lines.")
    print("summary_1 lines:", mid)
    print("summary_2 lines:", len(lines) - mid)
else:
    mid = (n + 1) // 2

    part1 = prefix + "".join(blocks[:mid])
    part2 = "".join(blocks[mid:])

    OUT1.write_text(part1)
    OUT2.write_text(part2)

    print("total paths:", n)
    print("summary_1 paths:", mid)
    print("summary_2 paths:", n - mid)
    print("wrote:", OUT1)
    print("wrote:", OUT2)

total paths: 29232
summary_1 paths: 14616
summary_2 paths: 14616
wrote: summary_1.txt
wrote: summary_2.txt


In [7]:
from pathlib import Path
import re

INPUT_FILE = Path("rule.txt")
OUTPUT_FILE = Path("rule_clean.txt")

text = INPUT_FILE.read_text(errors="ignore")

rule_pattern = re.compile(
    r"(?m)^RULE(?:\s+Path)?\s+\d+\s*:\s*"
)

matches = list(rule_pattern.finditer(text))

if not matches:
    raise ValueError("No RULE lines found in rule.txt")

blocks = []

for i, m in enumerate(matches):
    start = m.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    block = text[start:end].strip()
    blocks.append(block)

seen = set()
clean_rules = []

for block in blocks:
    body = re.sub(r"^RULE(?:\s+Path)?\s+\d+\s*:\s*", "", block, flags=re.IGNORECASE).strip()

    key = re.sub(r"\s+", " ", body).strip().lower()

    if key in seen:
        continue

    seen.add(key)
    clean_rules.append(body)

output_lines = []

for i, body in enumerate(clean_rules, 1):
    output_lines.append(f"RULE {i}: {body}")

OUTPUT_FILE.write_text("\n".join(output_lines) + "\n")

print("input rules:", len(blocks))
print("unique rules:", len(clean_rules))
print("removed duplicates:", len(blocks) - len(clean_rules))
print("wrote:", OUTPUT_FILE)

input rules: 29232
unique rules: 9116
removed duplicates: 20116
wrote: rule_clean.txt
